### Abhishek
### DSC 680, Week 8
### Generative AI-Powered Knowledge Management System for Codebase and Domain Expertise
### Leveraging LangChain’s Complete Documentation as Training Corpus
### Approach 1: Implemented LangChain Documentation in a Retrieval-Augmented Generation (RAG) Architecture
### Approach 2: Fine-Tuned Large Language Model Using LangChain Documentation as Training Data

## Approach 1: Implemented LangChain Documentation in a Retrieval-Augmented Generation (RAG) Architecture


1.  **`extract_text_from_html(file_path)`:** This function's purpose is to take raw HTML content from a given file and strip away all the HTML tags, leaving only the clean, readable text. This is a crucial first step, as LLMs typically process plain text, not formatted HTML.

In [12]:
from bs4 import BeautifulSoup
import tiktoken
import os
import pandas as pd
import openai
from openai import OpenAI # Import the new OpenAI client

# --- Function: extract_text_from_html ---

def extract_text_from_html(file_path):
    """
    Extract text from an HTML file.

    Parameters:
    - file_path (str): Path to the HTML file.

    Returns:
    - Extracted text from the HTML.
    """
    try:
        with open(file_path, 'r', encoding="utf-8") as file:
            content = file.read()
    except UnicodeDecodeError:
        # Fallback to latin-1 if utf-8 decoding fails
        with open(file_path, 'r', encoding="latin-1") as file:
            content = file.read()

    soup = BeautifulSoup(content, 'html.parser')

    # Extract and return text from the HTML content
    return soup.get_text()

2.  **`split_text_by_tokens(text, max_tokens)`:** Once the raw text is extracted, this function comes into play. LLMs often have limitations on the amount of text (measured in 'tokens') they can process at once. This function intelligently breaks down a long piece of text into smaller, manageable chunks, ensuring that each chunk does not exceed a specified `max_tokens` limit. This prevents overloading the language model and allows for processing very large documents incrementally.


In [13]:
# --- Function: split_text_by_tokens ---

def split_text_by_tokens(text, max_tokens=300):
    """
    Split text based on token count using the tiktoken library.

    Parameters:
    - text (str): Text to be split.
    - max_tokens (int): Maximum number of tokens per chunk.

    Returns:
    - List of text chunks, each not exceeding max_tokens.
    """

    # Get the encoder (tokenizer) for the specified model
    tokenizer = tiktoken.get_encoding('cl100k_base')

    # Split the text into chunks based on token count
    chunks = []
    current_chunk = ""
    for word in text.split():
        # Check if adding the word doesn't exceed the max_token count
        if len(tokenizer.encode(current_chunk + " " + word)) <= max_tokens:
            current_chunk += " " + word
        else:
            chunks.append(current_chunk.strip())
            current_chunk = word
    # Append any remaining text
    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks



3.  **`process_directory(directory, max_tokens)`:** This is the orchestrator. It automates the entire process for an entire collection of HTML files. It recursively searches through a specified directory (and its subdirectories), finds all HTML files, applies the `extract_text_from_html` function to each, and then uses `split_text_by_tokens` to segment the extracted text. The final output is a structured dictionary, mapping each original HTML file path to a list of its processed text chunks.

In [ ]:
# --- Function: process_directory ---

def process_directory(directory, max_tokens=400):
    """
    Process all HTML files within a directory (and its subdirectories), extracting text and splitting based on token count.

    Parameters:
    - directory (str): The root directory to start the search for HTML files.
    - max_tokens (int): Maximum number of tokens per chunk.

    Returns:
    - Dictionary with file paths as keys and lists of text chunks as values.
    """
    results = {}

    # Loop through the directory and its subdirectories
    for dirpath, dirnames, filenames in os.walk(directory):
        for filename in filenames:
            if filename.endswith('.html'):
                file_path = os.path.join(dirpath, filename)
                text = extract_text_from_html(file_path)
                chunks = split_text_by_tokens(text, max_tokens)
                results[file_path] = chunks


    return results


### `calculate_embeddings_for_dict` purpose

This function's main job is to take organized text chunks (from potentially many HTML files) and turn them into numerical vectors called **embeddings**. These embeddings are essential for tasks like semantic search, similarity comparisons, or even feeding into other machine learning models.

Imagine you have two very small 'files' (e.g., short documents) and each has been broken down into a few text `chunks`. The `calculate_embeddings_for_dict` function will:

1.  **Iterate through each 'file'**: It looks at `document1.html` and `document2.html` in our example.
2.  **Batch Chunks**: For each file, it groups its text chunks into 'batches' (e.g., up to 1000 chunks at a time to optimize API calls).
3.  **Send to OpenAI**: It sends these batches of text chunks to OpenAI's embedding model (`text-embedding-ada-002`).
4.  **Receive Embeddings**: OpenAI returns a list of numerical vectors (embeddings), one for each text chunk.
5.  **Store Results**: It collects all these embeddings and stores them back in a dictionary, keyed by the original file path. So, `document1.html` will now map to a list of its embeddings, and `document2.html` to its list.

The output will be a dictionary where each file path points to a list of lists, with each inner list being the numerical embedding for a text chunk from that file.

In [19]:
# --- Function: calculate_embeddings_for_dict ---

# Set an environment variable named 'API_KEY' with a placeholder for the OpenAI API key.
os.environ['API_KEY'] = 'sk-proj-R0t5EvB2oXzcdKBiWD2HvHjwwf0b1L9B9Qq7Uun8K7P971myu' + \
'6v1hwyjgDqvn1g1IxKdd2nqeFT3BlbkFJ4OKtM7LF7q-d4fiCFZJF3JN1LOYyQVSqpPvKH_kVt5539cUFNW5b7nguHjUzk8SCH8ZYi5FO4A'

def calculate_embeddings_for_dict(chunks_dict):
    """
    Calculate embeddings for a dictionary where each key is a file path and the
    corresponding value is a list of text chunks.

    Parameters:
    - chunks_dict (dict): Dictionary with file paths as keys and lists of text
      chunks as values.

    Returns:
    - DataFrame with file paths, all_chunks and their corresponding lists of
      embeddings.
    """
    # Initialize the OpenAI client
    client = OpenAI(api_key=os.getenv("API_KEY"))

    # Define the specific OpenAI embedding model to be used for generating embeddings.
    EMBEDDING_MODEL = "text-embedding-ada-002"
    # Define the number of text chunks to send to the OpenAI API in a single request.
    # This helps optimize API calls and stay within rate limits.
    BATCH_SIZE = 1000

    # Initialize lists to store flattened data for the DataFrame
    all_file_paths = []
    all_chunks_text = []
    all_embeddings_vectors = []
    file_counter = 0
    # Loop through each file path and its associated list of text chunks in the input
    # dictionary.
    for file_path, chunks in chunks_dict.items():
        embeddings_for_file = [] # Temporary list for embeddings of current file's chunks

        # Iterate through the chunks list in batches, defined by BATCH_SIZE.
        for batch_start in range(0, len(chunks), BATCH_SIZE):
            # Calculate the end index for the current batch.
            batch_end = batch_start + BATCH_SIZE
            # Extract the current batch of text chunks.
            batch = chunks[batch_start:batch_end]
            # Removed: print(f"Processing embeddings for {file_path}, batch {batch_start} to {batch_end-1}")
            # Call the OpenAI API to create embeddings for the current batch of text,
            # using the specified model and API key from environment variables.
            response = client.embeddings.create(model=EMBEDDING_MODEL, input=batch) # Updated API call
            # Iterate through the 'data' field in the API response, which contains the
            # embedding objects.
            for i, be in enumerate(response.data): # Access 'data' attribute
                # Assert that the index in the response matches the iteration index,
                # ensuring data integrity.
                assert i == be.index # Access 'index' attribute
            # Extract just the embedding vectors from each embedding object in the response.
            batch_embeddings = [e.embedding for e in response.data] # Access 'embedding' attribute
            # Add the embeddings from the current batch to the overall list of embeddings for
            # the file.
            embeddings_for_file.extend(batch_embeddings)

        # After processing all batches for a file, extend the global lists
        all_file_paths.extend([file_path] * len(chunks)) # Repeat file_path for each chunk
        all_chunks_text.extend(chunks) # Add all chunks for this file
        all_embeddings_vectors.extend(embeddings_for_file) # Add all embeddings for this file
        file_counter += 1
        print(f"Processed file {file_counter}: {file_path}") 
    # Return the dataframe containing file paths, all_chunks and their corresponding lists of
    # embeddings (one row per chunk).
    return pd.DataFrame({
        "file_path": all_file_paths,
        "text": all_chunks_text,
        "embeddings": all_embeddings_vectors
    })


In [ ]:
# Unzip the documentation.zip file quietly into the /content/documentation directory
print("Unzipping documentation.zip quietly...")
!unzip -q -o documentation.zip -d documentation
print("Unzipping complete.")

In [16]:
# --- Main Execution Blocks ---

# Assuming the unzipped files are in a directory named 'documentation'
documentation_directory = 'documentation'

# 1. Process the directory to extract text and split into chunks
print(f"Processing HTML files in {documentation_directory}...")
chunks_by_file = process_directory(documentation_directory)
print("Text extraction and splitting complete.")




Processing HTML files in documentation...
Text extraction and splitting complete.


In [17]:
max_chunk = 0
for k,v in chunks_by_file.items():
    max_chunk=len(v) if len(v) > max_chunk else max_chunk

print(f"Maximum number of chunks for any file: {max_chunk}")

Maximum number of chunks for any file: 86


In [18]:
len(chunks_by_file['documentation/apidocs/api.python.langchain.com/en/latest/experimental_api_reference.html'] )
    # Display the keys of the resulting dictionary to verify file paths

6

In [20]:
# Install PyTorch if missing
#%pip install torch --quiet

import torch

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

print('Device:', device)


Device: mps


In [21]:
# 2. Calculate embeddings for the processed chunks
print("Calculating embeddings for all text chunks...")
embeddings_dataframe = calculate_embeddings_for_dict(chunks_by_file)
print("Embeddings calculation complete.")



Calculating embeddings for all text chunks...
Processed file 1: documentation/__MACOSX/apidocs/api.python.langchain.com/en/latest/._index.html
Processed file 2: documentation/__MACOSX/apidocs/api.python.langchain.com/en/latest/._api_reference.html
Processed file 3: documentation/__MACOSX/apidocs/api.python.langchain.com/en/latest/._experimental_api_reference.html
Processed file 4: documentation/__MACOSX/apidocs/api.python.langchain.com/en/latest/chat_loaders/._langchain.chat_loaders.utils.map_ai_messages.html
Processed file 5: documentation/__MACOSX/apidocs/api.python.langchain.com/en/latest/chat_loaders/._langchain.chat_loaders.gmail.GMailLoader.html
Processed file 6: documentation/__MACOSX/apidocs/api.python.langchain.com/en/latest/chat_loaders/._langchain.chat_loaders.utils.merge_chat_runs_in_session.html
Processed file 7: documentation/__MACOSX/apidocs/api.python.langchain.com/en/latest/chat_loaders/._langchain.chat_loaders.telegram.TelegramChatLoader.html
Processed file 8: documen

In [38]:
# Display the first few rows of the resulting DataFrame
print("\nResulting Embeddings DataFrame (first 5 rows):")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
print(embeddings_dataframe.head(1).text)


str(embeddings_dataframe.head(1).text).split()



Resulting Embeddings DataFrame (first 5 rows):
0        Mac OS X         2   q      £                                      ATTR       £                               com.apple.provenance   Ç²õëx[O
Name: text, dtype: object


['0',
 '\x00\x05\x16\x07\x00\x02\x00\x00Mac',
 'OS',
 'X',
 '\x00\x02\x00\x00\x00',
 '\x00\x00\x002\x00\x00\x00q\x00\x00\x00\x02\x00\x00\x00£\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00ATTR\x00\x00\x00\x00\x00\x00\x00£\x00\x00\x00\x98\x00\x00\x00',
 '\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01\x00\x00\x00\x98\x00\x00\x00',
 '\x00\x00\x15com.apple.provenance\x00\x01\x00\x00Ç²õëx[O\x90',
 'Name:',
 'text,',
 'dtype:',
 'object']

In [39]:
# Display some information about the DataFrame
print(f"\nDataFrame shape: {embeddings_dataframe.shape}")
print("DataFrame info:")
embeddings_dataframe.info()


DataFrame shape: (12993, 3)
DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12993 entries, 0 to 12992
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   file_path   12993 non-null  object
 1   text        12993 non-null  object
 2   embeddings  12993 non-null  object
dtypes: object(3)
memory usage: 304.7+ KB


### Explanation of `search_similar_strings` Function

This function (`search_similar_strings`) is a key component for retrieving relevant information from a collection of documents (specifically, your `embeddings_dataframe`) by finding text chunks most semantically similar to a given search query.

Here's a breakdown of its parts:

*   **Purpose**: The primary goal is to find and return the `limit` number of text chunks from your `data_frame` that are most semantically similar to your `search_term` (your query).

*   **`from scipy import spatial`**: This line imports the `spatial` module from the `scipy` library. This module is used specifically for calculating the cosine distance between numerical vectors.

*   **`EMBEDDING_MODEL = "text-embedding-ada-002"`**: This variable stores the name of the OpenAI embedding model that will be used to convert the search query into its numerical embedding representation.

*   **`def search_similar_strings(...)`**: This is the function definition. It takes three parameters:
    *   `search_term` (string): The query you want to search for.
    *   `data_frame` (pandas DataFrame): This is your `embeddings_dataframe`, which contains the text chunks and their pre-calculated embeddings.
    *   `limit` (integer, default 100): The maximum number of most similar results you want to retrieve.

*   **`client = OpenAI(api_key=os.getenv("API_KEY"))`**: This line initializes the new OpenAI client using your API key from environment variables.

*   **`embedding_response = client.embeddings.create(model=EMBEDDING_MODEL, input=search_term)`**: This line makes an API call to OpenAI using the new client. It sends your `search_term` to the specified `EMBEDDING_MODEL` to get its numerical embedding.

*   **`search_embedding = embedding_response.data[0].embedding`**: From the API's response, this line extracts the actual numerical embedding vector (a list of numbers) for your `search_term`.

*   **`def similarity(x, y): return 1 - spatial.distance.cosine(x, y)`**: This inner function defines how similarity is measured. Cosine distance measures the angle between two vectors; a smaller angle (distance closer to 0) means higher dissimilarity. By subtracting it from 1, `similarity` becomes a score where 1 means perfectly similar, and 0 means completely dissimilar.

*   **`results = [...] for _, row in data_frame.iterrows()]`**: This is a list comprehension that performs the core comparison:
    1.  It iterates through each `row` in your `data_frame` (which contains your text chunks and their embeddings).
    2.  For each row, it calculates the `similarity` between the `search_embedding` (your query's embedding) and the `row["embeddings"]` (the embedding of that particular text chunk).
    3.  It stores a tuple `(row["text"], similarity_score)` for each comparison.

*   **`results.sort(key=lambda x: x[1], reverse=True)`**: This line sorts the `results` list. It sorts them based on the `similarity_score` (the second element of each tuple, `x[1]`) in descending order, so the most similar chunks appear first.

*   **`texts, scores = zip(*results)`**: After sorting, this line unpacks the list of `(text, score)` tuples into two separate lists: one for `texts` and one for `scores`.

*   **`return texts[:limit], scores[:limit]`**: Finally, the function returns the `limit` number of most similar text chunks and their corresponding similarity scores. This provides you with the most relevant document excerpts for your search query.

### Summary of `search_similar_strings`

In essence, `search_similar_strings` acts as a **semantic search engine**. It takes a natural language query, converts it into a numerical embedding using OpenAI, and then efficiently compares that query's embedding to a pre-computed database of document chunk embeddings. It then returns the most semantically related text chunks, ordered by their similarity, enabling a powerful way to retrieve relevant information from a large corpus based on meaning rather than just keywords.

In [40]:
from scipy import spatial  # for calculating vector similarities for search
import openai
from openai import OpenAI # Import the new OpenAI client
import os
import pandas as pd

EMBEDDING_MODEL = "text-embedding-ada-002"

def search_similar_strings(
    search_term: str,
    data_frame: pd.DataFrame,
    limit: int = 100
) -> tuple[list[str], list[float]]:
    """
    Search for strings in the data_frame that are most similar to the given search_term.

    Args:
    - search_term (str): The term to search for.
    - data_frame (pd.DataFrame): DataFrame containing strings and their embeddings.
    - limit (int): Maximum number of results to return.

    Returns:
    - tuple: Most similar strings and their similarity scores.
    """

    # Initialize the OpenAI client (assuming API_KEY is set as an environment variable)
    client = OpenAI(api_key=os.getenv("API_KEY"))

    # Calculate the embedding for the search term using the new client
    embedding_response = client.embeddings.create(model=EMBEDDING_MODEL, input=search_term)
    search_embedding = embedding_response.data[0].embedding # Access attributes directly

    # Define a function to calculate similarity between two embeddings
    def similarity(x, y):
        return 1 - spatial.distance.cosine(x, y)

    # Calculate similarities between search term and all strings in the dataframe
    results = [
        (row["text"], similarity(search_embedding, row["embeddings"])) # 'embeddings' column holds the list of embeddings
        for _, row in data_frame.iterrows()
    ]

    # Sort results by similarity score in descending order
    results.sort(key=lambda x: x[1], reverse=True)

    # Extract texts and their scores
    texts, scores = zip(*results)

    return texts[:limit], scores[:limit]

### Explanation of Core Functions: `num_tokens`, `query_message`, and `ask`

These three functions form the backbone of a "Retrieval Augmented Generation" (RAG) system, designed to intelligently answer questions by combining external knowledge with a Large Language Model (LLM).

#### 1. `num_tokens(text: str, model: str = GPT_MODEL) -> int`

*   **Purpose**: This helper function accurately calculates the number of tokens in a given `text` string using the `tiktoken` library, which is crucial for managing input size for LLMs.
*   **How it works**: LLMs don't process text directly; they break it down into smaller units called 'tokens' (which can be words, parts of words, or punctuation). Each LLM (or family of LLMs) has a specific way of doing this, defined by its tokenizer. This function uses `tiktoken.get_encoding(model)` to fetch the correct tokenizer for the specified `model` (e.g., `gpt-3.5-turbo-16k`). It then uses `encoding.encode(text)` to convert the text into a list of token IDs and returns the length of that list.
*   **Importance**: LLMs have strict limits on the number of tokens they can process in a single request. Using this function helps ensure that our generated prompts for the LLM stay within these limits, preventing errors and optimizing API usage.

#### 2. `query_message(query: str, df: pd.DataFrame, model: str) -> str`

*   **Purpose**: This function is responsible for creating the complete prompt that will be sent to the LLM. It augments the user's simple `query` with relevant information retrieved from your `embeddings_dataframe`.
*   **How it works**:
    1.  It first calls the `search_similar_strings` function (discussed previously) using the user's `query` and your `embeddings_dataframe` (`df`). This step identifies and retrieves the most semantically relevant text chunks from your documentation.
    2.  It prints the retrieved text chunks and their similarity scores for transparency and debugging.
    3.  It constructs an `introduction` for the LLM, clearly instructing it to use the provided documentation to answer the question, and to state if an answer cannot be found.
    4.  It concatenates the `strings` (the retrieved documentation chunks) into a single `documentation_content` block, typically separated by newlines.
    5.  It formats this content into a `documentation` string and appends the original `question`.
    6.  Finally, it combines these parts (`introduction + documentation + question`) into a single, comprehensive `message_content` string.
*   **Importance**: This is the core of Retrieval Augmented Generation (RAG). Instead of the LLM answering from its broad (but potentially outdated or unspecific) general knowledge, this function injects highly relevant, specific information from your documents directly into the prompt, leading to more accurate, contextual, and up-to-date answers.

#### 3. `ask(query: str, df: pd.DataFrame, model: str = GPT_MODEL, print_message: bool = False) -> str`

*   **Purpose**: This function orchestrates the entire process of sending a user's `query` to the LLM and returning its response, leveraging the context provided by `query_message`.
*   **How it works**:
    1.  `client = OpenAI(api_key=os.getenv("API_KEY"))`: Initializes the OpenAI API client. This client manages authentication using the `API_KEY` environment variable.
    2.  `message_content = query_message(query, df, model=model)`: Calls `query_message` to get the intelligently constructed prompt, including the relevant documentation.
    3.  `if print_message: print(message_content)`: (Optional) Allows you to see the exact prompt being sent to the LLM for debugging or understanding.
    4.  `messages = [...]`: Formats the `message_content` into the list of message objects (`role: system`, `role: user`) that the OpenAI Chat Completions API expects.
    5.  `response = client.chat.completions.create(...)`: Makes the actual API call to OpenAI. It sends the `model` (e.g., `gpt-3.5-turbo-16k`), the `messages` (your prompt), and a `temperature` setting (which controls the randomness of the LLM's output; `0` makes it more deterministic).
    6.  `response_message = response.choices[0].message.content`: Extracts the generated answer from the LLM's response object.
    7.  Returns the LLM's answer.
*   **Importance**: This function integrates all the components, from data retrieval to LLM interaction, providing a high-level interface to query your documentation-augmented LLM system.

### Why `GPT_MODEL = "gpt-3.5-turbo-16k"`?

*   `GPT_MODEL = "gpt-3.5-turbo-16k"` specifies the **Large Language Model** that will generate the answers. `gpt-3.5-turbo-16k` is a specific model from OpenAI's GPT-3.5 series that can process up to 16,000 tokens in a single request (both input and output). It's a balance of capability, speed, and cost for many applications. This is the model that *generates* human-like text responses.

### Is the embedding model and the GPT model related?

Yes, they are **related but serve different purposes** within this pipeline:

*   **Embedding Model (`text-embedding-ada-002`)**: This model's sole job is to convert text into numerical vectors (embeddings). These vectors capture the semantic meaning of the text. It's used for the *retrieval* part (finding relevant documents).
*   **GPT Model (`gpt-3.5-turbo-16k`)**: This is a conversational language model that *generates* human-like text. It takes the text (including the retrieved documentation and your question) and produces an answer. It's used for the *generation* part.

They are related because both are typically provided by the same AI service (like OpenAI) and are designed to work together in a RAG system. The embedding model helps the GPT model find the most relevant information to synthesize into an answer.

### How to decide which embedding model and which corresponding GPT model?

1.  **Embedding Model**:
    *   **Common Choice**: OpenAI's `text-embedding-ada-002` is a widely used and very capable general-purpose embedding model. It offers a good balance of performance and cost.
    *   **Considerations**:
        *   **Performance**: Newer models might offer better semantic understanding.
        *   **Cost**: Different models have different pricing per token.
        *   **Vector Database Compatibility**: Ensure the embedding model's output format is compatible with your chosen vector database (if you were to store these embeddings long-term).
        *   **Open-source Alternatives**: For local deployment or privacy, you might consider open-source embedding models (e.g., from Hugging Face) that can run on your own hardware.

2.  **GPT Model (or other LLMs)**:
    *   **GPT-3.5 Turbo (e.g., `gpt-3.5-turbo-16k`)**: Good for general-purpose question-answering, summarization, and instruction following. `16k` offers a larger context window, which is beneficial when injecting a lot of documentation.
    *   **GPT-4 (e.g., `gpt-4-turbo`)**: Offers superior reasoning, accuracy, and understanding, but is typically more expensive and slower. Use when accuracy and complex reasoning are paramount.
    *   **Smaller, Faster Models**: For simpler tasks, or when speed/cost are critical, smaller models might be sufficient.
    *   **Open-source LLMs (e.g., LLaMA, Mistral)**: Can be fine-tuned for specific tasks and run on your own infrastructure, offering more control and privacy, but require more computational resources to host.
    *   **Considerations**:
        *   **Context Window**: The maximum number of tokens (input + output) the model can handle. Crucial for RAG, as your injected documentation counts towards this.
        *   **Reasoning Capability**: How well the model can understand complex instructions and synthesize information.
        *   **Cost & Speed**: Trade-offs are common.
        *   **Fine-tuning**: Whether you need to fine-tune the model for very specific tasks or domains.

In essence, you choose an embedding model that accurately captures the semantics of your document content, and you choose a GPT model (or another LLM) that has the right balance of reasoning capability, context window size, speed, and cost for generating the final answer based on the retrieved information.

### Summary: The Crux of this RAG Pipeline

This entire system is designed to create an intelligent question-answering agent over your custom documentation. It works by:

1.  **Preparation**: First, it processes your raw HTML documentation by extracting clean text and intelligently splitting it into smaller, manageable chunks (using `extract_text_from_html` and `split_text_by_tokens`).
2.  **Indexing**: Then, for each of these text chunks, it generates a numerical representation called an 'embedding' (using `calculate_embeddings_for_dict` and the `text-embedding-ada-002` model). These embeddings are like detailed semantic fingerprints of each text chunk.
3.  **Retrieval (Semantic Search)**: When you ask a `query`, the system quickly converts *your query* into an embedding. It then compares this query embedding to all the document chunk embeddings (using `search_similar_strings`) to find the most semantically similar pieces of your documentation.
4.  **Augmentation & Generation**: Finally, it takes your original query and *augments* it with these retrieved, relevant documentation chunks. This combined prompt is then sent to a powerful LLM (like `gpt-3.5-turbo-16k`) (using `query_message` and `ask`). The LLM uses this specific context to generate a precise and informed answer, rather than relying solely on its general training data.

**Usage**: This pipeline is incredibly useful for building chatbots, knowledge bases, or search systems that can provide highly accurate and contextual answers based on your private or domain-specific data, overcoming the limitations of an LLM's general knowledge or cutoff dates.

In [47]:
import tiktoken
import pandas as pd
import openai
from openai import OpenAI # Import the new OpenAI client
import os # Import os for os.getenv

GPT_MODEL = "gpt-3.5-turbo-16k"

def num_tokens(text: str, model: str = GPT_MODEL) -> int:
    """Return the number of tokens in a string."""
    encoding = tiktoken.encoding_for_model(model)
    return len(encoding.encode(text))


def query_message(
    query: str,
    df: pd.DataFrame,
    model: str,
) -> str:
    """Return a message for GPT, with relevant source texts pulled from a dataframe."""
    # Reduce the limit of retrieved strings to fit within the model's context window
    strings, relatednesses = search_similar_strings(query, df, limit=20) # Reduced limit to 10
    print("Retrieved strings for query:")
    for s, r in zip(strings, relatednesses):
        print(f"  Similarity: {r:.4f}, Text: {s[:100]}...") # Print first 100 chars

    introduction = '''Use the below documentation on LangChain to answer the subsequent question.
    If the answer cannot be found in the documents, write "I could not find an answer."'''

    # Concatenate the relevant strings into the documentation part
    documentation_content = "\n".join(strings) # Join relevant strings with newlines
    documentation = f"Here's the documentation:\n{documentation_content}"

    question = f"\n\nQuestion: {query}"

    # Combine all parts to form the final message for the LLM
    return introduction + documentation + question

def ask(
    query: str,
    df: pd.DataFrame, # Expect df to be passed explicitly, no default global df
    model: str = GPT_MODEL,
    print_message: bool = False,
) -> str:
    """Answers a query using GPT and a dataframe of relevant texts and embeddings."""
    # Initialize the OpenAI client
    client = OpenAI(api_key=os.getenv("API_KEY"))

    message_content = query_message(query, df, model=model) # Corrected function call
    if print_message:
        print(message_content)
    messages = [
        {"role": "system", "content": "You answer questions about Langchain"},
        {"role": "user", "content": message_content},
    ]
    response = client.chat.completions.create( # Updated API call
        model=model,
        messages=messages,
        temperature=0
    )
    response_message = response.choices[0].message.content # Access attributes directly
    return response_message

In [48]:
# Define a sample query for testing
sample_query = "is RAG and LANGCHAIN same?"

# 1. Test num_tokens function
print("--- Testing num_tokens function ---")
sample_text_for_tokens = "This is a test sentence to count tokens."
tokens_count = num_tokens(sample_text_for_tokens)
print(f"Sample Text: '{sample_text_for_tokens}'")
print(f"Number of tokens: {tokens_count}")
print("---\n")

# 2. Test query_message function
print("--- Testing query_message function ---")
query_msg_output = query_message(sample_query, embeddings_dataframe, model=GPT_MODEL)
print(f"Query Message Output (first 500 chars):\n{query_msg_output[:500]}...")
print("---\n")

# 3. Test ask function
print("--- Testing ask function ---")
ask_output = ask(sample_query, embeddings_dataframe, print_message=True)
print(f"\nAsk Function Output:\n{ask_output}")
print("---\n")

--- Testing num_tokens function ---
Sample Text: 'This is a test sentence to count tokens.'
Number of tokens: 9
---

--- Testing query_message function ---
Retrieved strings for query:
  Similarity: 0.7858, Text: langchain.llms.ctranslate2 langchain.llms.databricks langchain.llms.deepinfra langchain.llms.deepspa...
  Similarity: 0.7815, Text: langchain.chains.graph_qa.neptune_cypher langchain.chains.graph_qa.sparql langchain.chains.hyde.base...
  Similarity: 0.7779, Text: langchain.retrievers.bm25 langchain.retrievers.chaindesk langchain.retrievers.chatgpt_plugin_retriev...
  Similarity: 0.7770, Text: langchain.callbacks.infino_callback langchain.callbacks.labelstudio_callback langchain.callbacks.llm...
  Similarity: 0.7754, Text: langchain.vectorstores.supabase langchain.vectorstores.tair langchain.vectorstores.tencentvectordb l...
  Similarity: 0.7716, Text: langchain.agents.agent_toolkits.sql.toolkit langchain.agents.agent_toolkits.vectorstore.base langcha...
  Similarity: 0.7709, T

## Approach 2: Fine-Tuned Large Language Model Using LangChain Documentation as Training Data

> Add blockquote


- I will repurpose my existing HTML processing functions (extract_text_from_html, split_text_by_tokens, process_directory) to clean and chunk the LangChain docs as my fine-tuning corpus.  
- I will pick a suitable base LLM (e.g., Llama, Mistral, T5, GPT-2 on Hugging Face) based on my task (Q&A, summarization, etc.) and my available compute.  
- I will convert the cleaned chunks into a proper fine-tuning dataset by creating structured, task-oriented examples (for example, question–answer pairs derived from the documentation).  
- I will implement the fine-tuning loop: load the pretrained model and tokenizer, load my dataset, set training hyperparameters (epochs, learning rate, batch size), and train using something like the Hugging Face Transformers Trainer or a custom loop.  
- I will evaluate the fine-tuned model on a held-out test set to check generalization, then package and deploy it so I can use it for inference on new LangChain-related queries.



In [50]:
# Install the Hugging Face Transformers library
#%pip install transformers torch accelerate

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# --- Choose a suitable base LLM ---
# For a beginner-friendly example, especially on free Colab tiers,
# a smaller model like GPT-2 is a good starting point.
# For production or more complex tasks, you'd consider Llama-2-7b, Mistral-7b, etc.

model_name = "gpt2" # Using GPT-2 as an example

print(f"Loading model: {model_name}")

# 1. Load the tokenizer for the chosen model
tokenizer = AutoTokenizer.from_pretrained(model_name)
print(f"Tokenizer for {model_name} loaded successfully.")

# 2. Load the pre-trained model
# AutoModelForCausalLM is suitable for generative tasks like Q&A
model = AutoModelForCausalLM.from_pretrained(model_name)
print(f"Model {model_name} loaded successfully.")

# Optional: Move model to GPU if available for faster processing
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print(f"Model moved to: {device}")

print("\n--- Model and Tokenizer Details ---")
print(f"Model architecture: {model.__class__.__name__}")
print(f"Tokenizer vocabulary size: {len(tokenizer)}")
print(f"Model number of parameters: {model.num_parameters()}")


Loading model: gpt2
Tokenizer for gpt2 loaded successfully.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model gpt2 loaded successfully.
Model moved to: cpu

--- Model and Tokenizer Details ---
Model architecture: GPT2LMHeadModel
Tokenizer vocabulary size: 50257
Model number of parameters: 124439808


## Data Preparation

### Subtask:
Extract text chunks from the `embeddings_dataframe` and tokenize them using the pre-loaded GPT-2 tokenizer, preparing them for a fine-tuning dataset.


**Reasoning**:
To prepare the data for fine-tuning, I need to extract the text chunks from the `embeddings_dataframe` and tokenize them using the pre-loaded GPT-2 tokenizer, ensuring proper padding and truncation.



In [ ]:
print("--- Data Preparation: Tokenizing text chunks ---")

# 1. Extract the 'text' column from the embeddings_dataframe
text_chunks = embeddings_dataframe['text'].tolist()
print(f"Extracted {len(text_chunks)} text chunks from embeddings_dataframe.")

# 2. Ensure the pre-loaded tokenizer has a padding token set
# If it doesn't, set it to the eos_token (end-of-sequence token).
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print("Tokenizer's pad_token set to eos_token.")

# 3. Tokenize the extracted text chunks
# Use padding and truncation enabled for consistent input to the model.
# The `return_tensors="pt"` argument makes the tokenizer return PyTorch tensors.
tokenized_dataset = tokenizer(text_chunks, padding=True, truncation=True, return_tensors="pt")

print("Text chunks tokenized successfully.")
print(f"Shape of input_ids: {tokenized_dataset['input_ids'].shape}")
print(f"Shape of attention_mask: {tokenized_dataset['attention_mask'].shape}")

**Reasoning**:
After tokenizing the text chunks, the next step is to convert them into a Hugging Face `Dataset` object, which is the standard format for training models with the Transformers library.



In [ ]:
from datasets import Dataset

print("--- Creating Hugging Face Dataset ---")

# Convert tokenized_dataset (which contains 'input_ids' and 'attention_mask' as tensors)
# into a Hugging Face Dataset. This format is required for the Trainer API.
dataset = Dataset.from_dict({
    'input_ids': tokenized_dataset['input_ids'],
    'attention_mask': tokenized_dataset['attention_mask'],
})

print("Hugging Face Dataset created successfully.")
print(f"Dataset features: {dataset.features}")
print(f"Dataset size: {len(dataset)}")
print(dataset)


**Reasoning**:
To fully prepare the dataset for causal language model fine-tuning, the `labels` column needs to be explicitly set to the `input_ids`, as the model will use these to learn next-token prediction.



In [ ]:
print("--- Preparing dataset for Causal Language Modeling ---")

# For Causal Language Models, the labels are typically the input_ids themselves.
# The model internally shifts the labels for next-token prediction.
def set_labels(examples):
    examples["labels"] = examples["input_ids"]
    return examples

# Apply the function to the dataset
dataset = dataset.map(set_labels, batched=True)

print("Labels column added to the dataset.")
print(f"Dataset features after adding labels: {dataset.features}")
print(f"First example labels: {dataset[0]['labels'][:10]}") # Print first 10 labels of the first example

**Reasoning**:
After preparing the dataset with labels, it's crucial to split it into training and validation sets to ensure proper evaluation of the fine-tuned model and prevent overfitting. This is a standard practice in machine learning.



In [ ]:
print("--- Splitting dataset into training and validation sets ---")

# Split the dataset into training and validation sets
# A common split ratio is 80% for training and 20% for validation.
train_test_split = dataset.train_test_split(test_size=0.2)

train_dataset = train_test_split['train']
val_dataset = train_test_split['test']

print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print("Dataset split successfully.")

**Reasoning**:
Now that the dataset is prepared and split, the next step is to set up the training configuration for the language model using `TrainingArguments` from the Hugging Face Transformers library. This defines hyperparameters and other training specifics.



## Adjust Training Arguments for Faster Training

### Subtask:
Modify the `TrainingArguments` to reduce the `num_train_epochs` to 1.


**Reasoning**:
To achieve faster training as requested by the subtask, I will modify the `num_train_epochs` parameter within the `TrainingArguments` to 1.



In [ ]:
from transformers import TrainingArguments, Trainer

print("--- Reconfiguring Training Arguments for faster training ---")

# Define the training arguments
training_args = TrainingArguments(
    output_dir="./gpt2_finetuned",
    num_train_epochs=1, # Reduced to 1 for faster training
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    report_to=["tensorboard"], # Use report_to instead of logging_dir for tensorboard integration
    logging_steps=100,
    save_strategy="epoch",
    load_best_model_at_end=False,
)


print("Training arguments reconfigured successfully.")
print(f"Output directory: {training_args.output_dir}")
print(f"Number of epochs: {training_args.num_train_epochs}")

## Start the Fine-tuning Process (Retry)

### Subtask:
Re-initialize the Hugging Face Trainer with the updated training arguments and then call the `train()` method to begin fine-tuning.


**Reasoning**:
Re-initialize the Trainer with the updated training arguments and then call the train() method to begin the fine-tuning process.



In [ ]:
print("--- Re-initializing Hugging Face Trainer and starting fine-tuning ---")

# Re-instantiate the Trainer with the updated training arguments
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print("Trainer re-initialized successfully.")

# Call the train() method on the Trainer object to begin fine-tuning
trainer.train()

print("Fine-tuning process started.")

# Task
Fine-tune a GPT-2 model on the LangChain documentation, then save the fine-tuned model and its tokenizer to "./gpt2_finetuned", evaluate its performance on a validation set, and finally, summarize the fine-tuning process, outcomes, and next steps for utilizing the model for Q&A on the LangChain documentation.

## Save the Fine-tuned Model

### Subtask:
After training is complete, save the fine-tuned model and its tokenizer to the specified output directory (./gpt2_finetuned).


**Reasoning**:
After the fine-tuning process, the model and tokenizer need to be saved to the specified output directory to preserve the trained state for future use.



In [ ]:
print("--- Saving fine-tuned model and tokenizer ---")

# Save the fine-tuned model
trainer.save_model(training_args.output_dir)
print(f"Model saved to {training_args.output_dir}")

# Save the tokenizer
tokenizer.save_pretrained(training_args.output_dir)
print(f"Tokenizer saved to {training_args.output_dir}")

**Reasoning**:
After saving the model, the next step is to evaluate its performance on the validation set to understand how well it generalizes.



In [ ]:
print("--- Evaluating model performance on the validation set...")

eval_results = trainer.evaluate(eval_dataset=val_dataset)

print("Evaluation results:")
for key, value in eval_results.items():
    print(f"  {key}: {value}")
print("Evaluation complete.")

## Generate Sample Q&A Prompts

### Subtask:
Create a Python list of dictionaries, where each dictionary contains a 'question' about LangChain and an 'expected_answer' that could be derived from its documentation.


**Reasoning**:
To fulfill the subtask, I will create a Python list of dictionaries named `qa_prompts`, populate it with at least 5 question-answer pairs related to LangChain, and then print the list to demonstrate its content.



In [ ]:
print("--- Generating Sample Q&A Prompts ---")

# 1. Define a Python list named qa_prompts
qa_prompts = [
    {
        'question': 'What is an LLMChain?',
        'expected_answer': 'An LLMChain is the simplest way to combine an LLM with a prompt template.'
    },
    {
        'question': 'What is LangChain?',
        'expected_answer': 'LangChain is a framework for developing applications powered by large language models (LLMs).'
    },
    {
        'question': 'How can I install LangChain?',
        'expected_answer': 'LangChain can be installed using pip: `pip install langchain`.'
    },
    {
        'question': 'What is a PromptTemplate in LangChain?',
        'expected_answer': 'A PromptTemplate is a reusable template for generating prompts to an LLM.'
    },
    {
        'question': 'What are Agents in LangChain?',
        'expected_answer': 'Agents involve an LLM making decisions about which Actions to take, observing an Observation, and repeating until the task is complete.'
    }
]

# 5. Print the qa_prompts list to display the generated Q&A pairs.
print("Generated Q&A Prompts:")
for item in qa_prompts:
    print(f"Question: {item['question']}")
    print(f"Expected Answer: {item['expected_answer']}")
    print("---")
print("Sample Q&A prompts generated successfully.")

# Task
Load the fine-tuned GPT-2 model and its tokenizer from the `./gpt2_finetuned` directory, then define a Python function to perform inference by taking a question, tokenizing it, generating a response from the model, and decoding the output. Finally, process the `qa_prompts` list using this inference function to generate and print model answers alongside the original questions and expected answers. Summarize the testing process and observations.

## Load Fine-tuned Model and Tokenizer

### Subtask:
Load the fine-tuned GPT-2 model and its tokenizer from the `./gpt2_finetuned` directory.


**Reasoning**:
Load the fine-tuned model and tokenizer from the specified directory, move the model to the appropriate device, and set it to evaluation mode as instructed.



In [ ]:
print("--- Loading fine-tuned model and tokenizer for inference ---")

# 1. Import AutoModelForCausalLM and AutoTokenizer from the transformers library.
from transformers import AutoModelForCausalLM, AutoTokenizer

# 2. Load the fine-tuned model
fine_tuned_model = AutoModelForCausalLM.from_pretrained('./gpt2_finetuned')
print("Fine-tuned model loaded successfully.")

# 3. Load the corresponding tokenizer
fine_tuned_tokenizer = AutoTokenizer.from_pretrained('./gpt2_finetuned')
print("Fine-tuned tokenizer loaded successfully.")

# 4. Move the fine_tuned_model to the appropriate device
# The 'device' variable should be available from previous steps (e.g., 'cuda' or 'cpu').
fine_tuned_model.to(device)
print(f"Fine-tuned model moved to: {device}")

# 5. Set the fine_tuned_model to evaluation mode
fine_tuned_model.eval()
print("Fine-tuned model set to evaluation mode.")

**Reasoning**:
Now that the fine-tuned model and tokenizer are loaded, the next step is to define a Python function for inference, as per the main task. This function will take a question, tokenize it, generate a response from the model, and then decode the output.



In [ ]:
print("--- Defining inference function for the fine-tuned model ---")

def generate_answer(question: str, model, tokenizer, max_length: int = 100) -> str:
    """
    Generates an answer to a given question using the fine-tuned GPT-2 model.

    Args:
        question (str): The input question.
        model: The fine-tuned causal language model.
        tokenizer: The tokenizer corresponding to the model.
        max_length (int): The maximum length of the generated answer.

    Returns:
        str: The generated answer.
    """

    # Encode the question into input IDs and move to the appropriate device
    input_ids = tokenizer.encode(question, return_tensors='pt').to(device)

    # Generate a response from the model
    # Use pad_token_id to ensure consistent generation behavior when padding is needed
    output = model.generate(
        input_ids,
        max_length=max_length + len(input_ids[0]), # Max length for generated part + input part
        num_return_sequences=1,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True, # Allow for more varied responses
        top_k=50, # Consider top 50 most likely next tokens
        top_p=0.95, # Consider tokens that cumulative probability is up to 95%
        temperature=0.7 # Control randomness
    )

    # Decode the generated tokens, skipping special tokens
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

    # Remove the original question from the generated text to get just the answer
    # This assumes the model completes the prompt after the question.
    if generated_text.startswith(question):
        answer = generated_text[len(question):].strip()
    else:
        answer = generated_text.strip()

    return answer

print("Inference function 'generate_answer' defined successfully.")

**Reasoning**:
Now that the inference function is defined, I will iterate through the `qa_prompts` list, generate an answer for each question using the `generate_answer` function, and then print the question, expected answer, and the model's generated answer for comparison.



In [ ]:
print("--- Processing Q&A prompts with the fine-tuned model ---")

model_answers = []

for i, qa_pair in enumerate(qa_prompts):
    question = qa_pair['question']
    expected_answer = qa_pair['expected_answer']

    print(f"\nProcessing prompt {i+1}/{len(qa_prompts)}")
    print(f"Question: {question}")
    print(f"Expected Answer: {expected_answer}")

    # Generate answer using the fine-tuned model
    # max_length is set to a reasonable value for short answers, adjust if needed
    generated_answer = generate_answer(question, fine_tuned_model, fine_tuned_tokenizer, max_length=150)
    model_answers.append(generated_answer)

    print(f"Model's Answer: {generated_answer}")
    print("--------------------------------------------------")

print("Finished processing all Q&A prompts.")

**Reasoning**:
The previous output showed a warning regarding the attention mask not being set during generation. To resolve this and ensure reliable results, I will modify the `generate_answer` function to explicitly pass the `attention_mask` to the `model.generate()` method.



In [ ]:
print("--- Defining inference function for the fine-tuned model ---")

def generate_answer(question: str, model, tokenizer, max_length: int = 100) -> str:
    """
    Generates an answer to a given question using the fine-tuned GPT-2 model.

    Args:
        question (str): The input question.
        model: The fine-tuned causal language model.
        tokenizer: The tokenizer corresponding to the model.
        max_length (int): The maximum length of the generated answer.

    Returns:
        str: The generated answer.
    """

    # Encode the question into input IDs and move to the appropriate device
    # Explicitly include attention_mask here
    encoded_input = tokenizer(question, return_tensors='pt').to(device)
    input_ids = encoded_input['input_ids']
    attention_mask = encoded_input['attention_mask']

    # Generate a response from the model
    # Use pad_token_id to ensure consistent generation behavior when padding is needed
    output = model.generate(
        input_ids,
        attention_mask=attention_mask, # Pass the attention mask here
        max_length=max_length + len(input_ids[0]), # Max length for generated part + input part
        num_return_sequences=1,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True, # Allow for more varied responses
        top_k=50, # Consider top 50 most likely next tokens
        top_p=0.95, # Consider tokens that cumulative probability is up to 95%
        temperature=0.7 # Control randomness
    )

    # Decode the generated tokens, skipping special tokens
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

    # Remove the original question from the generated text to get just the answer
    # This assumes the model completes the prompt after the question.
    if generated_text.startswith(question):
        answer = generated_text[len(question):].strip()
    else:
        answer = generated_text.strip()

    return answer

print("Inference function 'generate_answer' defined successfully.")

**Reasoning**:
With the `generate_answer` function now corrected to include the `attention_mask`, I will re-process the `qa_prompts` list to observe the model's answers with the improved inference function and verify that the warning is gone.



In [ ]:
print("--- Processing Q&A prompts with the fine-tuned model (re-run) ---")

model_answers = []

for i, qa_pair in enumerate(qa_prompts):
    question = qa_pair['question']
    expected_answer = qa_pair['expected_answer']

    print(f"\nProcessing prompt {i+1}/{len(qa_prompts)}")
    print(f"Question: {question}")
    print(f"Expected Answer: {expected_answer}")

    # Generate answer using the fine-tuned model
    # max_length is set to a reasonable value for short answers, adjust if needed
    generated_answer = generate_answer(question, fine_tuned_model, fine_tuned_tokenizer, max_length=150)
    model_answers.append(generated_answer)

    print(f"Model's Answer: {generated_answer}")
    print("--------------------------------------------------")

print("Finished processing all Q&A prompts (re-run).")

## Final Task

### Subtask:
Summarize the testing process and observations.


## Summary:

### Q&A
The testing process involved loading a fine-tuned GPT-2 model and its tokenizer, defining an inference function that explicitly handled the `attention_mask`, and then iterating through a set of five LangChain-related Q&A prompts. The model's generated answers were qualitatively compared against expected answers.

### Data Analysis Key Findings
*   The fine-tuned GPT-2 model, after just one epoch of training, demonstrated limited factual accuracy, often failing to provide concise and correct answers directly matching the expected responses.
*   The model exhibited significant repetitiveness and a tendency to hallucinate information, such as irrelevant details like "temperature in Boise, Idaho," or repeating phrases like `langchain.callbacks.chain.callbacks.chain...`.
*   For specific questions like "How can I install LangChain?", the model provided a completely incorrect and complex description instead of the simple `pip install langchain` instruction.
*   The model's responses often showed a contextual shift, diverging into general discussions rather than precise definitions relevant to the questions, indicating an inability to retain specific domain knowledge effectively.
*   Warnings related to the `attention_mask` during inference were successfully suppressed after an explicit correction in the `generate_answer` function.

### Insights or Next Steps
*   Increase the number of training epochs (`num_train_epochs`) to allow the model to learn more effectively and improve factual accuracy and coherence.
*   Consider using larger and more capable pre-trained models (e.g., Llama-2-7b, Mistral-7b) if computational resources permit, as GPT-2's capacity may be insufficient for complex technical Q&A tasks.
